# SSPH de la PES del modelo Ace-Gly-NMe

Este notebook calcula la homología persistente de conjuntos de subnivel de la superficie de energía potencial bidimensional del modelo de dipéptido de glicina **Ace-Gly-NMe** en fase gaseosa.

Correcciones incorporadas:

1. elimina los extremos periódicos duplicados antes de construir el complejo;
2. verifica cuantitativamente la continuidad de ambas costuras;
3. construye un `PeriodicCubicalComplex` directamente desde los valores de los vértices;
4. aplica condiciones periódicas en las dos coordenadas torsionales;
5. calcula (H_0), (H_1) y (H_2) sin forzar manualmente ninguna clase;
6. verifica que los números de Betti finales sean ((1,2,1));
7. exporta intervalos, resumen topológico, rejilla depurada y figuras para el artículo.

Ejecuta las celdas en orden. El archivo de entrada debe tener tres columnas sin encabezado: `phi`, `psi` y energía relativa en kcal/mol.

In [ ]:
# 1. Importaciones y configuración
from pathlib import Path
from importlib.metadata import version as package_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

import gudhi

print("GUDHI:", package_version("gudhi"))
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

# Archivo de entrada. Si no existe, se busca automáticamente morse_input*.txt.
INPUT_FILE = Path("data/processed/morse_input_periodic_symmetry_repaired.txt")
OUTPUT_PREFIX = "glycine_ssph"

# Homología con coeficientes en F_2.
HOMOLOGY_FIELD = 2

# Umbral usado para los resultados principales.
# El notebook también conserva todas las barras de persistencia positiva.
MIN_PERSISTENCE = 0.045  # kcal/mol

# Una discrepancia mayor que este valor genera una advertencia de costura.
SEAM_WARNING_TOL = 0.10  # kcal/mol

# Convención periódica: conservar el extremo inferior y retirar el superior.
# Ejemplo: conservar 180 grados y retirar 540 grados.
DROP_UPPER_PERIODIC_ENDPOINT = True

# Visualización. El clipping NUNCA se usa para calcular persistencia.
VISUAL_CLIP_MAX = 10.8  # usa None para mostrar todo el intervalo energético
ENERGY_CMAP = cm.turbo

# Paneles de subnivel. None selecciona niveles desde los eventos persistentes.
PANEL_LEVELS = None
N_PANELS = 16

SAVE_PNG_DPI = 300

In [ ]:
# 2. Lectura, validación y construcción de la rejilla periódica única
def resolve_input_file(preferred: Path) -> Path:
    preferred = Path(preferred)

    if not preferred.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo requerido: {preferred.resolve()}"
        )

    print(f"Archivo seleccionado de forma estricta: {preferred.resolve()}")
    return preferred


def read_energy_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=r"\s+",
        names=["phi_deg", "psi_deg", "energy_kcal_mol"],
        header=None,
        comment="#",
    )
    for column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna().reset_index(drop=True)
    if df.empty:
        raise ValueError("El archivo de energía no contiene datos numéricos válidos.")
    if df.duplicated(["phi_deg", "psi_deg"]).any():
        raise ValueError("Existen coordenadas exactamente duplicadas antes del ajuste periódico.")
    return df


def rectangular_grid(df: pd.DataFrame):
    phis = np.sort(df["phi_deg"].unique())
    psis = np.sort(df["psi_deg"].unique())
    E = (
        df.pivot(index="phi_deg", columns="psi_deg", values="energy_kcal_mol")
        .reindex(index=phis, columns=psis)
        .to_numpy(float)
    )
    if np.isnan(E).any():
        raise ValueError("La tabla no forma una rejilla rectangular completa.")
    return phis, psis, E


def seam_report(phis, psis, E):
    report = {}
    if np.isclose(phis[-1] - phis[0], 360.0):
        diff = np.abs(E[0, :] - E[-1, :])
        report["phi"] = {
            "mean_abs_diff": float(diff.mean()),
            "max_abs_diff": float(diff.max()),
            "coordinate_at_max": float(psis[np.argmax(diff)]),
        }
    if np.isclose(psis[-1] - psis[0], 360.0):
        diff = np.abs(E[:, 0] - E[:, -1])
        report["psi"] = {
            "mean_abs_diff": float(diff.mean()),
            "max_abs_diff": float(diff.max()),
            "coordinate_at_max": float(phis[np.argmax(diff)]),
        }
    return report


def print_seam_report(report):
    print("\nDiagnóstico de costuras equivalentes:")
    for direction, values in report.items():
        print(
            f"  {direction}: media={values['mean_abs_diff']:.6f}, "
            f"máximo={values['max_abs_diff']:.6f} kcal/mol, "
            f"coordenada transversal={values['coordinate_at_max']:.1f} grados"
        )
        if values["max_abs_diff"] > SEAM_WARNING_TOL:
            print(
                "    ADVERTENCIA: los extremos equivalentes convergieron a energías "
                "diferentes. Se conservará únicamente el extremo inferior."
            )


def remove_periodic_endpoint(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()
    for angle in ["phi_deg", "psi_deg"]:
        values = np.sort(clean[angle].unique())
        span = values[-1] - values[0]
        if np.isclose(span, 360.0):
            if not DROP_UPPER_PERIODIC_ENDPOINT:
                raise ValueError(
                    f"{angle} contiene ambos extremos periódicos. "
                    "Activa DROP_UPPER_PERIODIC_ENDPOINT."
                )
            clean = clean[~np.isclose(clean[angle], values[-1])].copy()

    clean["phi_deg"] = np.mod(clean["phi_deg"], 360.0)
    clean["psi_deg"] = np.mod(clean["psi_deg"], 360.0)

    if clean.duplicated(["phi_deg", "psi_deg"]).any():
        duplicates = clean[clean.duplicated(["phi_deg", "psi_deg"], keep=False)]
        raise ValueError(
            "Persisten coordenadas duplicadas después de retirar los extremos:\n"
            + duplicates.head(10).to_string(index=False)
        )
    return clean.sort_values(["phi_deg", "psi_deg"]).reset_index(drop=True)


input_path = resolve_input_file(INPUT_FILE)
raw_df = read_energy_table(input_path)
raw_phi, raw_psi, raw_E = rectangular_grid(raw_df)

print("Archivo:", input_path)
print("Registros brutos:", len(raw_df))
print("Rejilla bruta:", raw_E.shape)
print("Rango phi:", (raw_phi.min(), raw_phi.max()))
print("Rango psi:", (raw_psi.min(), raw_psi.max()))

seams = seam_report(raw_phi, raw_psi, raw_E)
print_seam_report(seams)

periodic_df = remove_periodic_endpoint(raw_df)
phi, psi, E_abs = rectangular_grid(periodic_df)
E = E_abs - float(np.min(E_abs))

if E.shape != (36, 36):
    raise ValueError(f"Se esperaba una rejilla periódica 36x36 y se obtuvo {E.shape}.")
if len(periodic_df) != 1296:
    raise ValueError(f"Se esperaban 1296 configuraciones únicas y hay {len(periodic_df)}.")

imin = np.unravel_index(np.argmin(E), E.shape)
imax = np.unravel_index(np.argmax(E), E.shape)

print("\nRejilla periódica definitiva:", E.shape)
print("Configuraciones únicas:", len(periodic_df))
print(
    "Mínimo global:", float(E[imin]), "kcal/mol en",
    (float(phi[imin[0]]), float(psi[imin[1]]))
)
print(
    "Máximo global:", float(E[imax]), "kcal/mol en",
    (float(phi[imax[0]]), float(psi[imax[1]]))
)

In [ ]:
# 3. Homología persistente periódica mediante lower-star cubical
def compute_periodic_persistence(E, field=2):
    complex_ = gudhi.PeriodicCubicalComplex(
        vertices=np.asfortranarray(E, dtype=float),
        periodic_dimensions=[True, True],
    )

    # 0.0 conserva todas las barras de persistencia estrictamente positiva.
    diagram = complex_.persistence(
        homology_coeff_field=int(field),
        min_persistence=0.0,
    )

    intervals = {}
    for dim in (0, 1, 2):
        arr = complex_.persistence_intervals_in_dimension(dim)
        if arr.size == 0:
            arr = np.empty((0, 2), dtype=float)
        intervals[dim] = np.asarray(arr, dtype=float).reshape(-1, 2)

    betti = tuple(int(x) for x in complex_.betti_numbers()[:3])
    return complex_, diagram, intervals, betti


def filter_intervals(intervals, min_persistence):
    filtered = {}
    for dim, arr in intervals.items():
        if len(arr) == 0:
            filtered[dim] = arr.copy()
            continue
        lifetime = arr[:, 1] - arr[:, 0]
        keep = np.isinf(arr[:, 1]) | (lifetime >= min_persistence)
        filtered[dim] = arr[keep]
    return filtered


pcc, diagram_all, intervals_all, betti = compute_periodic_persistence(
    E,
    field=HOMOLOGY_FIELD,
)
intervals = filter_intervals(intervals_all, MIN_PERSISTENCE)

print("Números de Betti del complejo completo:", betti)
if betti != (1, 2, 1):
    raise RuntimeError(
        f"El dominio periódico no recuperó la topología del toro: {betti} != (1, 2, 1)."
    )

for dim in (0, 1, 2):
    n_all = len(intervals_all[dim])
    n_report = len(intervals[dim])
    n_essential = int(np.isinf(intervals[dim][:, 1]).sum()) if n_report else 0
    print(
        f"H{dim}: {n_all} barras de persistencia positiva; {n_report} con persistencia >= "
        f"{MIN_PERSISTENCE}; {n_essential} esenciales"
    )

In [ ]:
# 4. Resumen Morse y exportación de resultados numéricos
def interval_dataframe(arr, dim):
    rows = []
    for birth, death in arr:
        rows.append(
            {
                "dimension": dim,
                "birth_kcal_mol": float(birth),
                "death_kcal_mol": float(death),
                "lifetime_kcal_mol": (
                    np.inf if np.isinf(death) else float(death - birth)
                ),
                "essential": bool(np.isinf(death)),
            }
        )
    return pd.DataFrame(rows)


def morse_counts(intervals):
    H0, H1, H2 = intervals[0], intervals[1], intervals[2]
    finite_H0 = int(np.isfinite(H0[:, 1]).sum()) if len(H0) else 0
    finite_H1 = int(np.isfinite(H1[:, 1]).sum()) if len(H1) else 0

    m0 = len(H0)
    m1 = finite_H0 + len(H1)
    m2 = finite_H1 + len(H2)
    return int(m0), int(m1), int(m2)


m0_all, m1_all, m2_all = morse_counts(intervals_all)
m0, m1, m2 = morse_counts(intervals)
euler = m0 - m1 + m2

print("\nEventos antes de aplicar el umbral:")
print("  índice 0, 1 y 2:", (m0_all, m1_all, m2_all))
print("\nEventos críticos visibles por persistencia:")
print("  mínimos (índice 0):", m0)
print("  sillas (índice 1):", m1)
print("  máximos (índice 2):", m2)
print("  relación de Euler:", f"{m0} - {m1} + {m2} = {euler}")

if euler != 0:
    raise RuntimeError("Los conteos filtrados no satisfacen chi(T^2)=0.")

for dim in (0, 1, 2):
    table = interval_dataframe(intervals[dim], dim)
    table.to_csv(f"{OUTPUT_PREFIX}_H{dim}.tsv", sep="\t", index=False)
    interval_dataframe(intervals_all[dim], dim).to_csv(
        f"{OUTPUT_PREFIX}_H{dim}_all_positive.tsv", sep="\t", index=False
    )

summary = pd.DataFrame(
    [
        {"quantity": "unique_phi_values", "value": len(phi)},
        {"quantity": "unique_psi_values", "value": len(psi)},
        {"quantity": "unique_configurations", "value": len(periodic_df)},
        {"quantity": "grid_step_deg", "value": float(np.median(np.diff(phi)))},
        {"quantity": "minimum_energy_kcal_mol", "value": float(E.min())},
        {"quantity": "maximum_energy_kcal_mol", "value": float(E.max())},
        {"quantity": "minimum_persistence_kcal_mol", "value": MIN_PERSISTENCE},
        {"quantity": "betti_0", "value": betti[0]},
        {"quantity": "betti_1", "value": betti[1]},
        {"quantity": "betti_2", "value": betti[2]},
        {"quantity": "morse_index_0_all_positive", "value": m0_all},
        {"quantity": "morse_index_1_all_positive", "value": m1_all},
        {"quantity": "morse_index_2_all_positive", "value": m2_all},
        {"quantity": "morse_index_0", "value": m0},
        {"quantity": "morse_index_1", "value": m1},
        {"quantity": "morse_index_2", "value": m2},
        {"quantity": "euler_characteristic", "value": euler},
    ]
)
summary.to_csv(f"{OUTPUT_PREFIX}_summary.tsv", sep="\t", index=False)

periodic_export = periodic_df.copy()
periodic_export["energy_kcal_mol"] -= periodic_export["energy_kcal_mol"].min()
periodic_export.to_csv(
    f"{OUTPUT_PREFIX}_periodic_grid.tsv",
    sep="\t",
    index=False,
)

display(summary)
for dim in (0, 1, 2):
    print(f"\nH{dim}")
    display(interval_dataframe(intervals[dim], dim))

In [ ]:
# 5. Funciones de visualización
COLORS = {0: "#d62f2f", 1: "#2b6cb0", 2: "#2ea44f"}


def closed_periodic_grid(phi, psi, E):
    phi_closed = np.r_[phi, 360.0]
    psi_closed = np.r_[psi, 360.0]
    E_closed = np.pad(E, ((0, 1), (0, 1)), mode="wrap")
    return phi_closed, psi_closed, E_closed


def visual_energy(E):
    if VISUAL_CLIP_MAX is None:
        return E.copy()
    return np.minimum(E, VISUAL_CLIP_MAX)


def representative_levels(intervals, n_panels, energy_max):
    if PANEL_LEVELS is not None:
        levels = np.asarray(PANEL_LEVELS, dtype=float)
    else:
        events = [float(E.min()), float(energy_max)]
        for arr in intervals.values():
            if len(arr):
                events.extend(arr[:, 0][np.isfinite(arr[:, 0])].tolist())
                events.extend(arr[:, 1][np.isfinite(arr[:, 1])].tolist())
        events = np.array(sorted(set(np.round(events, 10))), dtype=float)
        if len(events) <= n_panels:
            levels = events
        else:
            idx = np.unique(np.rint(np.linspace(0, len(events) - 1, n_panels)).astype(int))
            levels = events[idx]
        # Hace visibles las primeras cuencas y asegura el llenado final.
        if len(levels) > 1 and np.isclose(levels[0], E.min()):
            levels[0] = min(levels[1] * 0.25, 0.4)
        levels[-1] = energy_max
    return np.sort(np.unique(levels))


def plot_barcode(intervals, levels):
    fig, ax = plt.subplots(figsize=(13, 6))
    finite_deaths = []
    for arr in intervals.values():
        if len(arr):
            finite_deaths.extend(arr[:, 1][np.isfinite(arr[:, 1])])
    xmax = max([E.max(), *finite_deaths]) * 1.04

    lane = {0: 0.0, 1: 1.0, 2: 2.0}
    step = 0.075
    for dim in (0, 1, 2):
        arr = intervals[dim]
        finite = [(b, d) for b, d in arr if np.isfinite(d)]
        essential = [(b, d) for b, d in arr if np.isinf(d)]
        ordered = sorted(finite, key=lambda x: x[1] - x[0], reverse=True)
        y = lane[dim]
        for birth, death in ordered:
            ax.hlines(y, birth, death, color=COLORS[dim], linewidth=2.5)
            y += step
        for birth, _ in essential:
            ax.hlines(y, birth, xmax, color=COLORS[dim], linewidth=2.8)
            ax.annotate(
                "",
                xy=(xmax, y),
                xytext=(xmax - 0.015 * xmax, y),
                arrowprops={"arrowstyle": "->", "color": COLORS[dim], "lw": 2.0},
            )
            y += step

    for value in levels:
        ax.axvline(value, color="black", linewidth=0.6, alpha=0.18, zorder=0)

    ax.legend(
        handles=[Patch(color=COLORS[d], label=rf"$H_{d}$") for d in (0, 1, 2)],
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
    )
    ax.set_xlim(0, xmax)
    ax.set_ylim(-0.18, 2.65)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels([r"$H_0$", r"$H_1$", r"$H_2$"])
    ax.set_xlabel(r"Relative energy (kcal mol$^{-1}$)")
    ax.set_title("Sublevel-set persistent barcode")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_PREFIX}_barcode.png", dpi=SAVE_PNG_DPI, bbox_inches="tight")
    fig.savefig(f"{OUTPUT_PREFIX}_barcode.pdf", bbox_inches="tight")
    plt.show()


def plot_sublevel_panels(levels):
    phi_c, psi_c, E_c = closed_periodic_grid(phi, psi, E)
    X, Y = np.meshgrid(psi_c, phi_c)
    E_plot = visual_energy(E_c)

    ncols = 4
    nrows = int(np.ceil(len(levels) / ncols))
    fig = plt.figure(figsize=(15, 3.65 * nrows))
    gs = GridSpec(nrows, ncols, figure=fig, wspace=0.12, hspace=0.24)
    axes = []
    mappable = None

    for k, level in enumerate(levels):
        ax = fig.add_subplot(gs[k // ncols, k % ncols])
        axes.append(ax)
        mappable = ax.imshow(
            E_plot,
            origin="lower",
            extent=[0, 360, 0, 360],
            cmap=ENERGY_CMAP,
            vmin=0,
            vmax=(VISUAL_CLIP_MAX if VISUAL_CLIP_MAX is not None else E.max()),
            interpolation="nearest",
            aspect="equal",
        )
        if E.min() <= level <= E.max():
            ax.contour(X, Y, E_c, levels=[level], colors="black", linewidths=1.1)
        ax.set_title(f"{chr(65 + k)}: r = {level:.3f}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    for k in range(len(levels), nrows * ncols):
        ax = fig.add_subplot(gs[k // ncols, k % ncols])
        ax.axis("off")

    label = r"Relative energy (kcal mol$^{-1}$)"
    if VISUAL_CLIP_MAX is not None:
        label += f"; color clipped at {VISUAL_CLIP_MAX:g}"
    cbar = fig.colorbar(mappable, ax=axes, fraction=0.025, pad=0.025, location="left")
    cbar.set_label(label)
    fig.suptitle("Evolution of the sublevel sets", fontsize=18, y=0.995)
    fig.savefig(f"{OUTPUT_PREFIX}_sublevels.png", dpi=SAVE_PNG_DPI, bbox_inches="tight")
    fig.savefig(f"{OUTPUT_PREFIX}_sublevels.pdf", bbox_inches="tight")
    plt.show()


def plot_energy_landscape():
    phi_c, psi_c, E_c = closed_periodic_grid(phi, psi, E)
    X, Y = np.meshgrid(psi_c, phi_c)
    E_plot = visual_energy(E_c)
    vmax = VISUAL_CLIP_MAX if VISUAL_CLIP_MAX is not None else E.max()

    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(
        E_plot,
        origin="lower",
        extent=[0, 360, 0, 360],
        cmap=ENERGY_CMAP,
        vmin=0,
        vmax=vmax,
        interpolation="nearest",
        aspect="equal",
    )
    ax.plot(psi[imin[1]], phi[imin[0]], marker="x", color="white", ms=10, mew=2.2)
    ax.set_xlabel(r"$\psi$ (deg)")
    ax.set_ylabel(r"$\phi$ (deg)")
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_yticks(np.arange(0, 361, 60))
    cbar = fig.colorbar(image, ax=ax)
    cbar_label = r"Relative energy (kcal mol$^{-1}$)"
    if VISUAL_CLIP_MAX is not None:
        cbar_label += f"; clipped at {VISUAL_CLIP_MAX:g}"
    cbar.set_label(cbar_label)
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_PREFIX}_heatmap.png", dpi=SAVE_PNG_DPI, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 6))
    contour_max = VISUAL_CLIP_MAX if VISUAL_CLIP_MAX is not None else E.max()
    contour_levels = np.linspace(0, contour_max, 10)[1:]
    contours = ax.contour(X, Y, E_c, levels=contour_levels, cmap=ENERGY_CMAP)
    ax.clabel(contours, inline=True, fmt="%.1f", fontsize=8)
    ax.plot(psi[imin[1]], phi[imin[0]], "kx", ms=8, mew=1.8)
    ax.set_xlim(0, 360)
    ax.set_ylim(0, 360)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\psi$ (deg)")
    ax.set_ylabel(r"$\phi$ (deg)")
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_yticks(np.arange(0, 361, 60))
    ax.set_title("Gas-phase conformational energy landscape")
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_PREFIX}_contour.png", dpi=SAVE_PNG_DPI, bbox_inches="tight")
    plt.show()

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    surface = ax.plot_surface(
        X, Y, E_plot, cmap=ENERGY_CMAP, vmin=0, vmax=vmax,
        linewidth=0, antialiased=True
    )
    ax.set_xlabel(r"$\psi$ (deg)")
    ax.set_ylabel(r"$\phi$ (deg)")
    ax.set_zlabel(r"Relative energy (kcal mol$^{-1}$)")
    ax.set_xlim(0, 360)
    ax.set_ylim(0, 360)
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_yticks(np.arange(0, 361, 60))
    title = "Gas-phase energy surface"
    if VISUAL_CLIP_MAX is not None:
        title += f" (visual clip at {VISUAL_CLIP_MAX:g} kcal/mol)"
    ax.set_title(title)
    fig.colorbar(surface, ax=ax, shrink=0.62, pad=0.10).set_label(
        r"Relative energy (kcal mol$^{-1}$)"
    )
    ax.view_init(elev=34, azim=-58)
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_PREFIX}_surface3D.png", dpi=SAVE_PNG_DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# 6. Generación de las figuras definitivas
levels = representative_levels(intervals, N_PANELS, float(E.max()))

print("Niveles representativos:")
for letter, level in zip("ABCDEFGHIJKLMNOPQRSTUVWXYZ", levels):
    print(f"  {letter}: {level:.6f} kcal/mol")

plot_energy_landscape()
plot_sublevel_panels(levels)
plot_barcode(intervals, levels)

print("\nArchivos generados:")
for path in sorted(Path.cwd().glob(f"{OUTPUT_PREFIX}_*")):
    print(" ", path)

## Criterio para el artículo

Los conteos mostrados por el notebook después de aplicar `MIN_PERSISTENCE` deben describirse como **eventos críticos topológicamente resolubles a la escala de persistencia seleccionada**. Las barras eliminadas por el umbral deben conservarse en el análisis de sensibilidad y no interpretarse automáticamente como estados conformacionales químicamente distintos.

El mapa de colores puede truncarse para mejorar el contraste visual, pero la homología persistente siempre se calcula con la energía relativa completa y sin clipping.

## Relación entre el barcode y los paneles de subnivel

Las líneas verticales se etiquetan con las mismas letras A--P de los paneles.


In [ ]:
# 7. Código de barras con los niveles A--P de los paneles
def plot_barcode_with_sublevel_labels(intervals, levels):
    """Dibuja el barcode y relaciona cada línea vertical con un panel A, B, ..."""
    fig, ax = plt.subplots(figsize=(13, 6.4))

    finite_deaths = []
    for arr in intervals.values():
        if len(arr):
            finite_deaths.extend(arr[:, 1][np.isfinite(arr[:, 1])])
    xmax = max([float(E.max()), *finite_deaths]) * 1.04

    lane = {0: 0.0, 1: 1.0, 2: 2.0}
    step = 0.075
    for dim in (0, 1, 2):
        arr = intervals[dim]
        finite = [(b, d) for b, d in arr if np.isfinite(d)]
        essential = [(b, d) for b, d in arr if np.isinf(d)]
        ordered = sorted(finite, key=lambda pair: pair[1] - pair[0], reverse=True)
        y = lane[dim]

        for birth, death in ordered:
            ax.hlines(y, birth, death, color=COLORS[dim], linewidth=2.5)
            y += step

        for birth, _ in essential:
            ax.hlines(y, birth, xmax, color=COLORS[dim], linewidth=2.8)
            ax.annotate(
                "", xy=(xmax, y), xytext=(xmax - 0.015 * xmax, y),
                arrowprops={"arrowstyle": "->", "color": COLORS[dim], "lw": 2.0},
            )
            y += step

    # Líneas y letras que vinculan el barcode con los paneles de subnivel.
    for panel_index, value in enumerate(levels):
        letter = chr(65 + panel_index)
        ax.axvline(value, color="0.25", linewidth=0.65, alpha=0.35, zorder=0)
        close_group = [
            j for j, other in enumerate(levels)
            if abs(float(other) - float(value)) < 0.05
        ]
        rank = close_group.index(panel_index)
        horizontal_offset = 11.0 * (rank - 0.5 * (len(close_group) - 1))
        ax.annotate(
            letter,
            xy=(value, 2.58 - 0.13 * (panel_index % 2)),
            xytext=(horizontal_offset, 0),
            textcoords="offset points",
            ha="center",
            va="center",
            rotation=90,
            fontsize=7.8,
            fontweight="bold",
            bbox={
                "boxstyle": "round,pad=0.08",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.82,
            },
            zorder=6,
        )

    ax.legend(
        handles=[Patch(color=COLORS[d], label=rf"$H_{d}$") for d in (0, 1, 2)],
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
    )
    ax.set_xlim(0, xmax)
    ax.set_ylim(-0.18, 2.72)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels([r"$H_0$", r"$H_1$", r"$H_2$"])
    ax.set_xlabel(r"Relative energy (kcal mol$^{-1}$)")
    ax.set_title("Sublevel-set persistent barcode")
    ax.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(top=0.91, right=0.87)

    fig.savefig(
        f"{OUTPUT_PREFIX}_barcode_with_sublevels.png",
        dpi=SAVE_PNG_DPI,
        bbox_inches="tight",
    )
    fig.savefig(
        f"{OUTPUT_PREFIX}_barcode_with_sublevels.pdf",
        bbox_inches="tight",
    )
    plt.show()


plot_barcode_with_sublevel_labels(intervals, levels)


## Localización de los eventos críticos

Los marcadores M, S y X identifican, respectivamente, mínimos, sillas de índice 1 y máximos de índice 2. Cada etiqueta aparece en el primer panel cuyo umbral contiene el evento; en los paneles posteriores el marcador se conserva con menor opacidad.


In [ ]:
# 8. Puntos críticos en los paneles de subnivel
from matplotlib.lines import Line2D


def unused_grid_location(energy, used_locations, tolerance=1.0e-7):
    """Assign an unused grid vertex carrying the requested critical energy."""
    matches = np.argwhere(np.abs(E - energy) <= tolerance)
    if len(matches) == 0:
        nearest = np.unravel_index(np.argmin(np.abs(E - energy)), E.shape)
        error = float(abs(E[nearest] - energy))
        raise ValueError(
            f"No grid vertex matches {energy:.12f} kcal/mol; "
            f"nearest error = {error:.3e}."
        )

    for row, column in matches:
        location = (float(phi[row]), float(psi[column]))
        if location not in used_locations:
            used_locations.add(location)
            return location

    raise ValueError(
        f"All {len(matches)} vertices matching {energy:.12f} kcal/mol "
        "were already assigned to other critical events."
    )


def build_critical_event_table(intervals):
    """Construye la correspondencia Morse: mínimos, sillas y máximos."""
    rows = []

    # Índice 0: nacimientos de H0.
    for birth, _ in intervals[0]:
        rows.append((0, "Minimum", "Birth of H0", float(birth)))

    # Índice 1: muertes de H0 y nacimientos de H1.
    for _, death in intervals[0]:
        if np.isfinite(death):
            rows.append((1, "Index-1 saddle", "Death of H0", float(death)))
    for birth, _ in intervals[1]:
        rows.append((1, "Index-1 saddle", "Birth of H1", float(birth)))

    # Índice 2: muertes de H1 y nacimientos de H2.
    for _, death in intervals[1]:
        if np.isfinite(death):
            rows.append((2, "Index-2 maximum", "Death of H1", float(death)))
    for birth, _ in intervals[2]:
        rows.append((2, "Index-2 maximum", "Birth of H2", float(birth)))

    rows.sort(key=lambda row: (row[0], row[3], row[2]))
    counters = {0: 0, 1: 0, 2: 0}
    prefixes = {0: "M", 1: "S", 2: "X"}
    events = []
    used_locations = set()

    for morse_index, category, role, energy in rows:
        counters[morse_index] += 1
        phi_value, psi_value = unused_grid_location(energy, used_locations)
        events.append(
            {
                "event": f"{prefixes[morse_index]}{counters[morse_index]}",
                "morse_index": morse_index,
                "category": category,
                "topological_role": role,
                "energy_kcal_mol": energy,
                "phi_deg": phi_value,
                "psi_deg": psi_value,
            }
        )

    return pd.DataFrame(events)


critical_events = build_critical_event_table(intervals)
critical_events.to_csv(
    f"{OUTPUT_PREFIX}_critical_events.tsv", sep="\t", index=False
)


CRITICAL_STYLE = {
    0: {"marker": "o", "color": "#00E5FF", "label": "Minimum (M)"},
    1: {"marker": "^", "color": "#FF3EA5", "label": "Index-1 saddle (S)"},
    2: {"marker": "D", "color": "#FFD400", "label": "Index-2 maximum (X)"},
}


def periodic_marker_positions(value):
    # Un punto situado en 0 grados también se dibuja en 360 grados.
    return [0.0, 360.0] if np.isclose(value, 0.0) else [value]


def draw_critical_event(ax, event, is_new, add_label):
    style = CRITICAL_STYLE[int(event.morse_index)]
    size = 34 if is_new else 16
    alpha = 1.0 if is_new else 0.42
    edge_width = 0.90 if is_new else 0.55

    for x in periodic_marker_positions(float(event.psi_deg)):
        for y in periodic_marker_positions(float(event.phi_deg)):
            ax.scatter(
                x, y,
                s=size,
                marker=style["marker"],
                facecolor=style["color"],
                edgecolor="black",
                linewidth=edge_width,
                alpha=alpha,
                zorder=7,
                clip_on=True,
            )

    if not add_label:
        return

    x = float(event.psi_deg)
    y = float(event.phi_deg)
    if np.isclose(x, 0.0):
        x = 360.0
    if np.isclose(y, 0.0):
        y = 360.0

    dx = -5 if x >= 320 else 5
    dy = -5 if y >= 320 else 5
    ax.annotate(
        event.event,
        xy=(x, y),
        xytext=(dx, dy),
        textcoords="offset points",
        ha="right" if dx < 0 else "left",
        va="top" if dy < 0 else "bottom",
        fontsize=6.3,
        fontweight="bold",
        color="black",
        bbox={
            "boxstyle": "round,pad=0.12",
            "facecolor": "white",
            "edgecolor": "black",
            "linewidth": 0.35,
            "alpha": 0.88,
        },
        zorder=9,
        annotation_clip=False,
    )


def plot_sublevels_with_critical_points(levels, critical_events):
    phi_c, psi_c, E_c = closed_periodic_grid(phi, psi, E)
    X, Y = np.meshgrid(psi_c, phi_c)
    E_plot = visual_energy(E_c)
    vmax = VISUAL_CLIP_MAX if VISUAL_CLIP_MAX is not None else float(E.max())

    ncols = 4
    nrows = int(np.ceil(len(levels) / ncols))
    fig = plt.figure(figsize=(15.5, 3.95 * nrows))
    gs = GridSpec(nrows, ncols, figure=fig, wspace=0.12, hspace=0.27)
    axes = []
    mappable = None
    previous_level = -np.inf
    tolerance = 1.0e-7

    for panel_index, level in enumerate(levels):
        ax = fig.add_subplot(gs[panel_index // ncols, panel_index % ncols])
        axes.append(ax)
        mappable = ax.imshow(
            E_plot,
            origin="lower",
            extent=[0, 360, 0, 360],
            cmap=ENERGY_CMAP,
            vmin=0,
            vmax=vmax,
            interpolation="nearest",
            aspect="equal",
        )
        if float(E.min()) <= level <= float(E.max()):
            ax.contour(X, Y, E_c, levels=[level], colors="black", linewidths=1.05)

        visible = critical_events[
            critical_events["energy_kcal_mol"] <= level + tolerance
        ]
        for event in visible.itertuples(index=False):
            is_new = event.energy_kcal_mol > previous_level + tolerance
            draw_critical_event(ax, event, is_new=is_new, add_label=is_new)

        ax.set_title(f"{chr(65 + panel_index)}: r = {level:.3f}", fontsize=10.5)
        ax.set_xlim(0, 360)
        ax.set_ylim(0, 360)
        ax.set_xticks([])
        ax.set_yticks([])
        previous_level = level

    for panel_index in range(len(levels), nrows * ncols):
        ax = fig.add_subplot(gs[panel_index // ncols, panel_index % ncols])
        ax.axis("off")

    colorbar_label = r"Relative energy (kcal mol$^{-1}$)"
    if VISUAL_CLIP_MAX is not None:
        colorbar_label += f"; color clipped at {VISUAL_CLIP_MAX:g}"
    cbar = fig.colorbar(
        mappable, ax=axes, fraction=0.024, pad=0.025, location="left"
    )
    cbar.set_label(colorbar_label)

    legend_handles = [
        Line2D(
            [0], [0],
            marker=CRITICAL_STYLE[index]["marker"],
            linestyle="None",
            markerfacecolor=CRITICAL_STYLE[index]["color"],
            markeredgecolor="black",
            markeredgewidth=0.8,
            markersize=6.5,
            label=CRITICAL_STYLE[index]["label"],
        )
        for index in (0, 1, 2)
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.55, 0.012),
        ncol=3,
        frameon=False,
        fontsize=10,
    )
    fig.suptitle(
        "Evolution of the sublevel sets and critical events",
        fontsize=18,
        y=0.995,
    )
    fig.subplots_adjust(bottom=0.065)
    fig.savefig(
        f"{OUTPUT_PREFIX}_sublevels_critical_points.png",
        dpi=SAVE_PNG_DPI,
        bbox_inches="tight",
    )
    fig.savefig(
        f"{OUTPUT_PREFIX}_sublevels_critical_points_final.pdf",
        bbox_inches="tight",
    )
    plt.show()


plot_sublevels_with_critical_points(levels, critical_events)
display(critical_events)
